# Python Type Hints & mypy
Covers: basic hints, Optional/Union/Any, TypeVar/Generic, Protocol, Literal, TypedDict, overload, runtime checking, Python 3.10+ syntax

## 1. Basic Type Hints

In [ ]:
# Variable annotations
name: str = 'Alice'
age: int = 30
pi: float = 3.14159
active: bool = True
items: list[int] = [1, 2, 3]
config: dict[str, int] = {'timeout': 30, 'retries': 3}

# Function annotations
def greet(name: str, times: int = 1) -> str:
    return (f'Hello, {name}! ' * times).strip()

def add(a: int, b: int) -> int:
    return a + b

def log(message: str) -> None:
    print(f'[LOG] {message}')

print(greet('Alice'))
print(greet('Bob', 3))
print(add(2, 3))

# Type hints are NOT enforced at runtime!
result = add('hello', 'world')  # runs fine!
print('No runtime enforcement:', result)

# But you can check at runtime with isinstance
def safe_add(a: int, b: int) -> int:
    if not isinstance(a, int) or not isinstance(b, int):
        raise TypeError(f'Expected int, got {type(a).__name__} and {type(b).__name__}')
    return a + b

try:
    safe_add('hello', 'world')
except TypeError as e:
    print('TypeError:', e)

## 2. Optional, Union, and Python 3.10+ Syntax

In [ ]:
from typing import Optional, Union
import sys

# Optional[X] == Union[X, None]
def find_user(user_id: int) -> Optional[str]:
    users = {1: 'Alice', 2: 'Bob', 3: 'Charlie'}
    return users.get(user_id)

print('find_user(1):', find_user(1))
print('find_user(99):', find_user(99))

# Union — multiple types
def stringify(value: Union[int, float, str, bool]) -> str:
    return str(value)

# Python 3.10+ syntax (preferred)
if sys.version_info >= (3, 10):
    def find_user_new(user_id: int) -> str | None:
        users = {1: 'Alice', 2: 'Bob'}
        return users.get(user_id)
    
    def process(value: int | str | float) -> str:
        return str(value)
    
    print('\nPython 3.10+ union syntax: int | str | None')

# Type narrowing with isinstance
def handle(value: int | str) -> str:
    if isinstance(value, int):
        # mypy knows value is int here
        return f'Integer doubled: {value * 2}'
    # mypy knows value is str here
    return f'String upper: {value.upper()}'

print('\nType narrowing:')
print(handle(42))
print(handle('hello'))

## 3. TypeVar and Generic

In [ ]:
from typing import TypeVar, Generic, Sequence

T = TypeVar('T')
K = TypeVar('K')
V = TypeVar('V')

# Generic function
def first(items: list[T]) -> T | None:
    return items[0] if items else None

def last(items: Sequence[T]) -> T | None:
    return items[-1] if items else None

# mypy infers T from the argument
s: str | None = first(['a', 'b', 'c'])  # T = str
n: int | None = first([1, 2, 3])         # T = int
print('first(["a","b","c"]):', s)
print('first([1,2,3]):', n)

# Generic class
class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    
    def push(self, item: T) -> None:
        self._items.append(item)
    
    def pop(self) -> T:
        if not self._items:
            raise IndexError('Stack is empty')
        return self._items.pop()
    
    def peek(self) -> T | None:
        return self._items[-1] if self._items else None
    
    def __len__(self) -> int:
        return len(self._items)
    
    def __repr__(self) -> str:
        return f'Stack({self._items})'

# Type-safe usage
int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)
print('\nStack:', int_stack)
print('pop:', int_stack.pop())
print('peek:', int_stack.peek())

str_stack: Stack[str] = Stack()
str_stack.push('hello')
str_stack.push('world')
print('String stack:', str_stack)

# Generic with two type params
class Pair(Generic[K, V]):
    def __init__(self, key: K, value: V) -> None:
        self.key = key
        self.value = value
    
    def swap(self) -> 'Pair[V, K]':
        return Pair(self.value, self.key)
    
    def __repr__(self) -> str:
        return f'Pair({self.key!r}, {self.value!r})'

p: Pair[str, int] = Pair('age', 30)
print('\nPair:', p)
print('Swapped:', p.swap())

## 4. Protocol — Structural Subtyping

In [ ]:
from typing import Protocol, runtime_checkable

# Protocol — duck typing with static type checking
class Drawable(Protocol):
    def draw(self) -> str: ...

class Resizable(Protocol):
    def resize(self, factor: float) -> None: ...

class Shape(Drawable, Resizable, Protocol):
    """Combined protocol."""

# Classes don't inherit from Protocol!
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius
    
    def draw(self) -> str:
        return f'Circle(r={self.radius:.1f})'
    
    def resize(self, factor: float) -> None:
        self.radius *= factor

class Rectangle:
    def __init__(self, w: float, h: float) -> None:
        self.w = w
        self.h = h
    
    def draw(self) -> str:
        return f'Rectangle({self.w:.1f}x{self.h:.1f})'
    
    def resize(self, factor: float) -> None:
        self.w *= factor
        self.h *= factor

def render_all(shapes: list[Drawable]) -> None:
    for shape in shapes:
        print(' ', shape.draw())

def resize_all(shapes: list[Shape], factor: float) -> None:
    for shape in shapes:
        shape.resize(factor)

shapes = [Circle(5.0), Rectangle(4.0, 3.0), Circle(2.0)]
print('Before resize:')
render_all(shapes)

resize_all(shapes, 2.0)
print('After resize (2x):')
render_all(shapes)

# @runtime_checkable — enables isinstance
@runtime_checkable
class HasLen(Protocol):
    def __len__(self) -> int: ...

print('\nruntime_checkable isinstance:')
for obj in [[1, 2, 3], 'hello', (1, 2), {1, 2}, 42, None]:
    print(f'  {type(obj).__name__}: {isinstance(obj, HasLen)}')

## 5. Literal, TypedDict, overload

In [ ]:
from typing import Literal, TypedDict, overload

# Literal — restrict to specific values
Direction = Literal['north', 'south', 'east', 'west']
LogLevel = Literal['DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL']

def move(direction: Direction, steps: int = 1) -> str:
    return f'Moving {direction} {steps} step(s)'

def log(level: LogLevel, message: str) -> None:
    print(f'[{level}] {message}')

print(move('north', 3))
log('INFO', 'Application started')

# TypedDict — typed dictionary
class UserInfo(TypedDict):
    name: str
    age: int
    email: str

class UserInfoOptional(TypedDict, total=False):
    name: str
    age: int
    email: str

def process_user(user: UserInfo) -> str:
    return f"{user['name']} (age {user['age']}) <{user['email']}>"

user: UserInfo = {'name': 'Alice', 'age': 30, 'email': 'alice@example.com'}
print('\nTypedDict:', process_user(user))

# overload — different signatures for different types
@overload
def double(value: int) -> int: ...
@overload
def double(value: str) -> str: ...
@overload
def double(value: list) -> list: ...

def double(value):
    if isinstance(value, int):
        return value * 2
    elif isinstance(value, str):
        return value * 2
    elif isinstance(value, list):
        return value * 2
    raise TypeError(f'Unsupported type: {type(value).__name__}')

print('\noverload:')
print('double(5):', double(5))           # int
print('double("hi"):', double('hi'))     # str
print('double([1,2]):', double([1, 2]))  # list